# NB10 — Track B: joint fine-tuning of the hybrid (CAMeLBERT-MSA + Vstat), K=9

**End-to-end model.** CAMeLBERT-MSA is **trainable** (not frozen). Each article → sliding chunks
(≤510, stride 460), first **K=9** chunks (covers ~99% of articles fully), [CLS] per chunk →
**mean-pool** → Vneural∈R^768. Concatenate the 5 scaled stat features → **standardize the fused 773-vec**
(policy fixed with you) → MLP head (hidden 256, LayerNorm, dropout 0) → binary logit.

**Memory:** K=9 trainable chunks is heavy → **gradient checkpointing** + small batch + grad-accum 16.
**Resilience:** full **checkpoint/resume** — every epoch (and every N steps) saves model+optimizer+scaler+
scheduler+RNG+epoch to /kaggle/working; on restart it auto-resumes from the last checkpoint. Kaggle
Save-Version runs top-to-bottom, so if a commit dies mid-train, re-committing continues where it stopped
(point the resume path at the previous run's saved checkpoint dataset).

Smoke test on 40 articles first. Eval: std split + reduced LOGO (GPT + DeepSeek).

## 1 · Config

In [1]:
import os, math, random, numpy as np, pandas as pd, torch
P_DATASET  = "/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet"          # EDIT
P_VSTAT16  = "/kaggle/input/notebooks/bahaaqassem/ph2-nb6e-extract-11-features/vstat16_scaled.parquet"   # EDIT (5 new features live here, scaled)
CKPT_DIR   = "/kaggle/working/ckpt"                                # checkpoints saved here
# to RESUME a crashed run: add the previous run's output as a dataset and point here at its ckpt:
RESUME_FROM = "/kaggle/input/nb10-ckpt/ckpt"                       # EDIT or leave (auto-ignored if absent)

MODEL_ID   = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
STAT_COLS  = ["burstiness", "ttr", "quote_ratio", "function_word_ratio", "compressibility"]  # NEW_5
K_CHUNKS, MAX_CT, STRIDE = 9, 510, 460
HIDDEN, DROPOUT = 256, 0.0
LR_ENC, LR_HEAD, WD = 2e-5, 1e-3, 0.01
EPOCHS, MICRO_BS, GRAD_ACCUM = 3, 2, 16
SAVE_EVERY_STEPS = 200
SEED = 42
SMOKE = False          # <<< set False for the full run; True = 40-article dry run to check memory/plumbing
GENERATORS_LOGO = ["gpt", "deepseek"]     # reduced LOGO

os.makedirs(CKPT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("config loaded | device", DEV)

config loaded | device cuda


## 2 · Load + align data, build the fused-vector standardizer (train-fit)

In [2]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
v16 = pd.read_parquet(P_VSTAT16)
if "article_id" in v16.columns: v16 = v16.set_index("article_id")
df = df.loc[df.index.intersection(v16.index)]; v16 = v16.loc[df.index]
assert (df.index==v16.index).all()
assert not (set(STAT_COLS)-set(v16.columns)), "missing stat cols"

Xstat = v16[STAT_COLS].to_numpy(np.float32)
y     = df["label"].to_numpy(np.int64)
gen   = df["generator"].fillna("__human__").to_numpy()
split = df["split"].to_numpy()
texts = df["text"].astype(str).tolist()

if SMOKE:   # tiny balanced dry run
    idx = np.r_[np.where((split=="train")&(y==0))[0][:20], np.where((split=="train")&(y==1))[0][:20]]
    SMOKE_IDX = set(idx.tolist())
print(f"{len(df)} articles | stat dim {Xstat.shape[1]} | SMOKE={SMOKE}")

7101 articles | stat dim 5 | SMOKE=False


## 3 · Tokenize to first-K chunks (cached to disk once)

In [3]:
!pip install -q transformers
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id
CHUNK_LEN = MAX_CT + 2   # + [CLS] + [SEP]

def to_chunks(text):
    ids = tok(text, add_special_tokens=False)["input_ids"] or [tok.unk_token_id]
    wins = [ids[i:i+MAX_CT] for i in range(0, len(ids), STRIDE)][:K_CHUNKS] or [ids[:MAX_CT]]
    out = []
    for w in wins:
        seq = [CLS] + w + [SEP]
        seq = seq + [PAD]*(CHUNK_LEN-len(seq))
        out.append(seq)
    return out   # list of up to K chunks, each CHUNK_LEN long

CACHE_TOK = "/kaggle/working/chunks_K9.npz"
if os.path.exists(CACHE_TOK):
    z = np.load(CACHE_TOK, allow_pickle=True); CH = z["ch"]; NCH = z["nch"]
    print("loaded cached chunk ids")
else:
    from tqdm.auto import tqdm
    CH = np.full((len(texts), K_CHUNKS, CHUNK_LEN), PAD, dtype=np.int32)
    NCH = np.zeros(len(texts), np.int8)
    for i,t in enumerate(tqdm(texts, desc="chunking")):
        ch = to_chunks(t); NCH[i]=len(ch)
        for j,c in enumerate(ch): CH[i,j]=c
    np.savez_compressed(CACHE_TOK, ch=CH, nch=NCH)
    print("built + cached chunk ids ->", CACHE_TOK)
print("chunk tensor:", CH.shape)

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

chunking:   0%|          | 0/7101 [00:00<?, ?it/s]

built + cached chunk ids -> /kaggle/working/chunks_K9.npz
chunk tensor: (7101, 9, 512)


## 4 · Model: chunk-encoder (mean-pool over real chunks) + fused standardizer + MLP head

In [4]:
import torch.nn as nn
from transformers import AutoModel

class HybridModel(nn.Module):
    def __init__(self, stat_dim):
        super().__init__()
        self.enc = AutoModel.from_pretrained(MODEL_ID)
        self.enc.gradient_checkpointing_enable()          # <<< key memory saver for K=9 trainable
        h = self.enc.config.hidden_size                   # 768
        fused = h + stat_dim
        # standardizer for the FUSED vector (buffers filled from TRAIN before training)
        self.register_buffer("mu",  torch.zeros(fused))
        self.register_buffer("sd",  torch.ones(fused))
        self.head = nn.Sequential(
            nn.Linear(fused, HIDDEN), nn.LayerNorm(HIDDEN), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 2))
    def encode(self, ids, nch):
        # ids: (B, K, L)  nch: (B,) number of real chunks
        B,K,L = ids.shape
        flat = ids.view(B*K, L)
        att  = (flat != PAD).long()
        cls  = self.enc(input_ids=flat, attention_mask=att).last_hidden_state[:,0,:]  # (B*K, h)
        cls  = cls.view(B,K,-1)
        m = (torch.arange(K, device=ids.device)[None,:] < nch[:,None]).float().unsqueeze(-1)  # real-chunk mask
        return (cls*m).sum(1) / m.sum(1).clamp(min=1)     # mean-pool over real chunks
    def forward(self, ids, nch, stat):
        v = torch.cat([self.encode(ids, nch), stat], dim=1)
        v = (v - self.mu) / self.sd
        return self.head(v)
print("model defined")

model defined


## 5 · Fused-standardizer stats from TRAIN only (one frozen pass)

In [5]:
from torch.utils.data import DataLoader, Dataset
class DS(Dataset):
    def __init__(self, rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        i=self.rows[k]
        return (torch.from_numpy(CH[i].astype(np.int64)), int(NCH[i]),
                torch.from_numpy(Xstat[i]), int(y[i]), i)
def collate(b):
    ids=torch.stack([x[0] for x in b]); nch=torch.tensor([x[1] for x in b])
    st=torch.stack([x[2] for x in b]); yy=torch.tensor([x[3] for x in b]); idx=[x[4] for x in b]
    return ids,nch,st,yy,idx

train_rows = [i for i in range(len(df)) if split[i]=="train" and (not SMOKE or i in SMOKE_IDX)]

@torch.no_grad()
def fit_fused_standardizer(model):
    model.eval(); dl=DataLoader(DS(train_rows), batch_size=8, collate_fn=collate)
    from tqdm.auto import tqdm
    n=0; s=None; s2=None
    for ids,nch,st,yy,idx in tqdm(dl, desc="standardizer"):
        ids,nch,st=ids.to(DEV),nch.to(DEV),st.to(DEV)
        v=torch.cat([model.encode(ids,nch), st],1)
        s  = v.sum(0)  if s  is None else s +v.sum(0)
        s2 = (v*v).sum(0) if s2 is None else s2+(v*v).sum(0)
        n += v.shape[0]
    mu=s/n; sd=torch.sqrt(torch.clamp(s2/n - mu*mu, min=1e-6))
    model.mu.copy_(mu); model.sd.copy_(sd)
    return mu,sd
print("standardizer routine ready")

standardizer routine ready


## 6 · Checkpoint helpers (save/resume: model+opt+sched+scaler+RNG+epoch/step)

In [6]:
def save_ckpt(path, model, opt, sched, scaler, epoch, step):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"model":model.state_dict(), "opt":opt.state_dict(),
                "sched":sched.state_dict() if sched else None,
                "scaler":scaler.state_dict(), "epoch":epoch, "step":step,
                "torch_rng":torch.get_rng_state(), "cuda_rng":torch.cuda.get_rng_state_all(),
                "np_rng":np.random.get_state(), "py_rng":random.getstate()}, path)

def find_resume():
    for base in (CKPT_DIR, RESUME_FROM):
        p=os.path.join(base, "last.pt")
        if os.path.exists(p): return p
    return None

def load_ckpt(path, model, opt, sched, scaler):
    ck=torch.load(path, map_location=DEV)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    if sched and ck["sched"]: sched.load_state_dict(ck["sched"])
    scaler.load_state_dict(ck["scaler"])
    torch.set_rng_state(ck["torch_rng"].cpu()); torch.cuda.set_rng_state_all([s.cpu() for s in ck["cuda_rng"]])
    np.random.set_state(ck["np_rng"]); random.setstate(ck["py_rng"])
    print(f"resumed from {path}: epoch {ck['epoch']} step {ck['step']}")
    return ck["epoch"], ck["step"]
print("checkpoint helpers ready")

checkpoint helpers ready


## 7 · Train (grad-accum, gradient checkpointing, class-balanced, auto-resume)

In [7]:
from torch.utils.data import WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup

def build_optim(model, total_steps):
    enc_p = [p for n,p in model.named_parameters() if n.startswith("enc.")]
    head_p= [p for n,p in model.named_parameters() if not n.startswith("enc.")]
    opt = torch.optim.AdamW([{"params":enc_p,"lr":LR_ENC},{"params":head_p,"lr":LR_HEAD}], weight_decay=WD)
    sched = get_linear_schedule_with_warmup(opt, int(0.06*total_steps), total_steps)
    return opt, sched

def train():
    model = HybridModel(len(STAT_COLS)).to(DEV)
    cw = compute_class_weight("balanced", classes=np.array([0,1]), y=y[train_rows])
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEV))
    dl = DataLoader(DS(train_rows), batch_size=MICRO_BS, shuffle=True, collate_fn=collate, drop_last=True)
    total = (len(dl)//GRAD_ACCUM)*EPOCHS
    opt, sched = build_optim(model, total)
    scaler = torch.amp.GradScaler('cuda')

    resume = find_resume(); start_ep, gstep = 0, 0
    if resume: start_ep, gstep = load_ckpt(resume, model, opt, sched, scaler)
    else:
        fit_fused_standardizer(model)     # only on a fresh start
        save_ckpt(os.path.join(CKPT_DIR,"last.pt"), model, opt, sched, scaler, 0, 0)

    model.train()
    for ep in range(start_ep, EPOCHS):
        opt.zero_grad()
        for bi,(ids,nch,st,yy,idx) in enumerate(dl):
            ids,nch,st,yy = ids.to(DEV),nch.to(DEV),st.to(DEV),yy.to(DEV)
            with torch.amp.autocast('cuda'):
                logits = model(ids,nch,st); loss = lossf(logits, yy)/GRAD_ACCUM
            scaler.scale(loss).backward()
            if (bi+1)%GRAD_ACCUM==0:
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(); gstep+=1
                if gstep%SAVE_EVERY_STEPS==0:
                    save_ckpt(os.path.join(CKPT_DIR,"last.pt"), model, opt, sched, scaler, ep, gstep)
                    print(f"  ep{ep} step{gstep} loss {loss.item()*GRAD_ACCUM:.4f}")
        save_ckpt(os.path.join(CKPT_DIR,f"epoch{ep+1}.pt"), model, opt, sched, scaler, ep+1, gstep)
        save_ckpt(os.path.join(CKPT_DIR,"last.pt"),        model, opt, sched, scaler, ep+1, gstep)
        print(f"epoch {ep+1} done, saved.")
    return model

model = train()
print("training complete" if not SMOKE else "SMOKE run complete — set SMOKE=False for the real run")

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

standardizer:   0%|          | 0/671 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


epoch 1 done, saved.
  ep1 step200 loss 0.0025
epoch 2 done, saved.
  ep2 step400 loss 0.0003
epoch 3 done, saved.
training complete


## 8 · Evaluate: std split + reduced LOGO (GPT, DeepSeek)

In [8]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

@torch.no_grad()
def predict(model, rows):
    model.eval(); dl=DataLoader(DS(rows), batch_size=8, collate_fn=collate)
    P=[]; Y=[]
    for ids,nch,st,yy,idx in dl:
        ids,nch,st=ids.to(DEV),nch.to(DEV),st.to(DEV)
        with torch.amp.autocast('cuda'):
            p=torch.softmax(model(ids,nch,st),1)[:,1]
        P.append(p.float().cpu().numpy()); Y.append(yy.numpy())
    return np.concatenate(P), np.concatenate(Y)

if not SMOKE:
    te=[i for i in range(len(df)) if split[i]=="test"]
    p,yt=predict(model, te); pred=(p>=0.5).astype(int)
    print(f"STD  MacroF1 {100*f1_score(yt,pred,average='macro'):.2f}  "
          f"Acc {100*accuracy_score(yt,pred):.2f}  AUC {100*roc_auc_score(yt,p):.2f}")
    # NOTE: reduced LOGO here retrains per held-out generator; for the full study run it as its own
    # commit per fold (each fold = a fresh train() excluding that generator's AI). Placeholder summary:
    print("For LOGO(GPT,DeepSeek): run train() once per held-out generator (exclude its train+val AI),")
    print("evaluate its test-split AI + all test human. Keep the SAME K=9 + standardizer policy.")
else:
    print("SMOKE done — plumbing OK. Flip SMOKE=False, confirm GPU memory holds at K=9, then full run.")

STD  MacroF1 99.82  Acc 99.82  AUC 100.00
For LOGO(GPT,DeepSeek): run train() once per held-out generator (exclude its train+val AI),
evaluate its test-split AI + all test human. Keep the SAME K=9 + standardizer policy.
